In [3]:
!pip -q install datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 16.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 14.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.9/388.9 kB 13.8 MB/s eta 0:00:00


In [4]:
import time
import matplotlib as plt
import numpy as np
import pandas as pd
import copy
import re
import shelve

from datasets import load_dataset, Dataset

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Unhealthy Comments Corpus (UCC)

In [5]:
# Load Unhealthy Comments Corpus dataset
ucc_train = pd.read_csv('/content/drive/My Drive/Colab Notebooks/AISI_project/og_datasets/unhealthy_conversations_corpus/train.csv')
ucc_test = pd.read_csv('/content/drive/My Drive/Colab Notebooks/AISI_project/og_datasets/unhealthy_conversations_corpus/test.csv')
ucc_val = pd.read_csv('/content/drive/My Drive/Colab Notebooks/AISI_project/og_datasets/unhealthy_conversations_corpus/val.csv')

In [6]:
# Drop irrelevant columns:
ucc_train.drop(['generalisation', 'generalisation:confidence', 'generalisation_unfair', 'generalisation_unfair:confidence', 'healthy', 'healthy:confidence'], axis='columns', inplace=True)
ucc_test.drop(['generalisation', 'generalisation:confidence', 'generalisation_unfair', 'generalisation_unfair:confidence', 'healthy', 'healthy:confidence'], axis='columns', inplace=True)
ucc_val.drop(['generalisation', 'generalisation:confidence', 'generalisation_unfair', 'generalisation_unfair:confidence', 'healthy', 'healthy:confidence'], axis='columns', inplace=True)

print('train length: ', len(ucc_train))
print('test length: ', len(ucc_test))
print('validation length: ', len(ucc_val))

train length:  35503
test length:  4425
validation length:  4427


## Prepare UCC training dataset

In [7]:
ucc_train.head()

,_unit_id,_trusted_judgments,comment,antagonize,antagonize:confidence,condescending,condescending:confidence,dismissive,dismissive:confidence,hostile,hostile:confidence,sarcastic,sarcastic:confidence
0,2319157561,4,"Three marriages, several bankrupt periods, inh...",0,1.0000,0,1.0000,0,1.0000,0,0.7565,0,1.0000
1,1739464982,4,The sense of entitlement among high school 'jo...,0,0.7634,0,0.7634,0,0.7634,0,0.7634,0,0.7634
2,1739457583,5,So what? He was just stating the obvious.,0,0.8121,0,0.5928,0,0.8043,0,1.0000,0,1.0000
3,2319156950,40,"If one is a Con, why yes, one would honk. Loud...",0,0.8508,0,0.8867,0,0.9239,0,0.9641,0,0.8868
4,2327196492,3,Ooohhh... It's Wendy Whiner... making sure to ...,0,1.0000,0,1.0000,0,1.0000,0,1.0000,0,1.0000


In [8]:
attributes = ucc_train.columns[3:].to_list()

for attribute in attributes:
    if not 'confidence' in attribute:
        pos_df = ucc_train.loc[ucc_train[attribute] == 1]
        neg_df = ucc_train.loc[ucc_train[attribute] == 0]
        print("%s:\tPositive(1): %d\t(%d%%)\t\t Negative(0): %d\t(%d%%)"
              % (attribute, len(pos_df), 100*len(pos_df)/len(ucc_train), len(neg_df), 100*len(neg_df)/len(ucc_train)))

antagonize:	Positive(1): 1689	(4%)		 Negative(0): 33814	(95%)
condescending:	Positive(1): 1927	(5%)		 Negative(0): 33576	(94%)
dismissive:	Positive(1): 1071	(3%)		 Negative(0): 34432	(96%)
hostile:	Positive(1): 923	(2%)		 Negative(0): 34580	(97%)
sarcastic:	Positive(1): 1501	(4%)		 Negative(0): 34002	(95%)


In [9]:
np.random.seed(0)

# Resample UCC data to balance labels
ucc_antagonize = ucc_train.loc[ucc_train['antagonize'] == 1]
ucc_condescending = ucc_train.loc[ucc_train['condescending'] == 1]
ucc_dismissive = ucc_train.loc[ucc_train['dismissive'] == 1]
ucc_hostile = ucc_train.loc[ucc_train['hostile'] == 1]
ucc_sarcastic = ucc_train.loc[ucc_train['sarcastic'] == 1]

# Sample positive 'antagonize' instances
sample_indices = np.random.choice(len(ucc_antagonize), size=923, replace=False)
ucc_antagonize = ucc_antagonize.iloc[sample_indices]

# Sample positive 'condescending' instances
sample_indices = np.random.choice(len(ucc_condescending), size=923, replace=False)
ucc_condescending = ucc_condescending.iloc[sample_indices]

# Sample positive 'dismissive' instances
sample_indices = np.random.choice(len(ucc_dismissive), size=923, replace=False)
ucc_dismissive = ucc_dismissive.iloc[sample_indices]

# Sample positive 'sarcastic' instances
sample_indices = np.random.choice(len(ucc_sarcastic), size=923, replace=False)
ucc_sarcastic = ucc_sarcastic.iloc[sample_indices]

# Combine all sampled positive instances
ucc_train_balanced = pd.concat([ucc_antagonize, ucc_condescending, ucc_dismissive, ucc_hostile, ucc_sarcastic])
ucc_train_balanced.drop_duplicates(inplace=True)

# Re-examine balance of classes
for attribute in attributes:
    if not 'confidence' in attribute:
        pos_df = ucc_train_balanced.loc[ucc_train_balanced[attribute] == 1]
        neg_df = ucc_train_balanced.loc[ucc_train_balanced[attribute] == 0]
        print("%s:\tPositive(1): %d\t(%d%%)\t\t Negative(0): %d\t(%d%%)"
              % (attribute, len(pos_df), 100*len(pos_df)/len(ucc_train_balanced), len(neg_df), 100*len(neg_df)/len(ucc_train_balanced)))

antagonize:	Positive(1): 1467	(51%)		 Negative(0): 1405	(48%)
condescending:	Positive(1): 1575	(54%)		 Negative(0): 1297	(45%)
dismissive:	Positive(1): 1009	(35%)		 Negative(0): 1863	(64%)
hostile:	Positive(1): 923	(32%)		 Negative(0): 1949	(67%)
sarcastic:	Positive(1): 1130	(39%)		 Negative(0): 1742	(60%)


In [10]:
ucc_train_balanced.head()

,_unit_id,_trusted_judgments,comment,antagonize,antagonize:confidence,condescending,condescending:confidence,dismissive,dismissive:confidence,hostile,hostile:confidence,sarcastic,sarcastic:confidence
857,2254074833,17,The LEFT struggle with the TRUTH all the time....,1,0.5860,0,0.7800,0,0.7711,1,0.5856,0,0.8303
14440,2164632164,4,Bottom feeders ? It takes one to know one.,1,0.7591,1,1.0000,1,0.7591,1,0.7591,1,0.7517
23506,2028122331,5,Too funny... Not to a bright-eyed 'know-a-good...,1,0.6089,0,0.5930,0,0.5930,0,0.7893,0,0.7893
27637,1739472627,5,I hate the pope already,1,0.7906,0,1.0000,0,0.6019,0,0.6019,0,1.0000
25566,1739461243,5,"@DC Toronto - whoops, my bad! Change comment t...",1,0.7919,1,0.5885,1,0.5950,0,0.6111,1,0.7919


In [11]:
# Drop data instances with low confidence:
confident = ucc_train_balanced.loc[ucc_train_balanced['antagonize:confidence'] > 0.6]
confident = confident.loc[confident['condescending:confidence'] > 0.6]
confident = confident.loc[confident['dismissive:confidence'] > 0.6]
confident = confident.loc[confident['hostile:confidence'] > 0.6]
confident = confident.loc[confident['sarcastic:confidence'] > 0.6]
ucc_train_balanced_confident = confident

In [12]:
# Re-examine balance of classes
for attribute in attributes:
    if not 'confidence' in attribute:
        pos_df = ucc_train_balanced_confident.loc[ucc_train_balanced_confident[attribute] == 1]
        neg_df = ucc_train_balanced_confident.loc[ucc_train_balanced_confident[attribute] == 0]
        print("%s:\tPositive(1): %d\t(%d%%)\t\t Negative(0): %d\t(%d%%)"
              % (attribute, len(pos_df), 100*len(pos_df)/len(ucc_train_balanced_confident), len(neg_df), 100*len(neg_df)/len(ucc_train_balanced_confident)))

antagonize:	Positive(1): 412	(46%)		 Negative(0): 470	(53%)
condescending:	Positive(1): 454	(51%)		 Negative(0): 428	(48%)
dismissive:	Positive(1): 286	(32%)		 Negative(0): 596	(67%)
hostile:	Positive(1): 286	(32%)		 Negative(0): 596	(67%)
sarcastic:	Positive(1): 379	(42%)		 Negative(0): 503	(57%)


In [13]:
ucc_train_balanced_confident.head()

,_unit_id,_trusted_judgments,comment,antagonize,antagonize:confidence,condescending,condescending:confidence,dismissive,dismissive:confidence,hostile,hostile:confidence,sarcastic,sarcastic:confidence
14440,2164632164,4,Bottom feeders ? It takes one to know one.,1,0.7591,1,1.000,1,0.7591,1,0.7591,1,0.7517
27637,1739472627,5,I hate the pope already,1,0.7906,0,1.000,0,0.6019,0,0.6019,0,1.0000
15836,1739445421,3,"What's stupid is politicizing this, dumaz.",1,1.0000,1,1.000,1,1.0000,1,1.0000,0,1.0000
28468,1739456157,5,"Yeah, nothing like innocent until proven guilt...",1,0.8023,1,0.604,1,0.6040,1,0.6040,1,0.8017
6160,2327179282,5,'...the gay community tries to hijack every op...,1,0.6186,0,0.814,0,0.8140,0,0.7907,0,0.8140


In [14]:
print("balanced: ", len(ucc_train_balanced))
print("balanced and confidence > 0.6: ", len(ucc_train_balanced_confident))

balanced:  2872
balanced and confidence > 0.6:  882


In [15]:
# Balanced train dataset and Confident train dataset with confidence scores kept as features
ucc_train = ucc_train_balanced.drop(['_unit_id', '_trusted_judgments'], axis='columns')
ucc_train_confident = ucc_train_balanced_confident.drop(['_unit_id', '_trusted_judgments'], axis='columns')

# Balanced train dataset and Confident train dataset with confidence scores removed
ucc_train_no_scores = ucc_train_balanced.drop(['_unit_id', '_trusted_judgments', 'antagonize:confidence',
                                               'condescending:confidence', 'dismissive:confidence',
                                               'hostile:confidence', 'sarcastic:confidence'], axis='columns')

ucc_train_confident_no_scores = ucc_train_balanced_confident.drop(['_unit_id', '_trusted_judgments', 'antagonize:confidence',
                                               'condescending:confidence', 'dismissive:confidence',
                                               'hostile:confidence', 'sarcastic:confidence'], axis='columns')


In [16]:
# Save prepared Training datasets to files

ucc_train.to_csv('/content/drive/My Drive/Colab Notebooks/AISI_project/data/ucc_train.csv', index=False)
ucc_train_confident.to_csv('/content/drive/My Drive/Colab Notebooks/AISI_project/data/ucc_train_confident.csv', index=False)
ucc_train_no_scores.to_csv('/content/drive/My Drive/Colab Notebooks/AISI_project/data/ucc_train_no_scores.csv', index=False)
ucc_train_confident_no_scores.to_csv('/content/drive/My Drive/Colab Notebooks/AISI_project/data/ucc_train_confident_no_scores.csv', index=False)


In [17]:
# Re-examine distribution of training data
print('Total Training Instances:\t', len(ucc_train_no_scores))

ucc_antagonize = ucc_train_no_scores.loc[ucc_train_no_scores['antagonize'] == 1]
ucc_condescending = ucc_train_no_scores.loc[ucc_train_no_scores['condescending'] == 1]
ucc_dismissive = ucc_train_no_scores.loc[ucc_train_no_scores['dismissive'] == 1]
ucc_hostile = ucc_train_no_scores.loc[ucc_train_no_scores['hostile'] == 1]
ucc_sarcastic = ucc_train_no_scores.loc[ucc_train_no_scores['sarcastic'] == 1]

print('antagonize:\t', len(ucc_antagonize))
print('condescending:\t', len(ucc_condescending))
print('dismissive:\t', len(ucc_dismissive))
print('hostile:\t', len(ucc_hostile))
print('sarcastic:\t', len(ucc_sarcastic))

cpr = ucc_train_no_scores.loc[ucc_train_no_scores['dismissive'] == 0]
cpr = cpr.loc[cpr['antagonize'] == 0]
cpr = cpr.loc[cpr['hostile'] == 0]
cpr = cpr.loc[cpr['sarcastic'] == 0]
cpr = cpr.loc[cpr['condescending'] == 0]
print('Instances where all classes are negative:\t', len(cpr))

cpr = ucc_train_no_scores.loc[ucc_train_no_scores['dismissive'] == 1]
cpr = cpr.loc[cpr['antagonize'] == 1]
cpr = cpr.loc[cpr['hostile'] == 1]
cpr = cpr.loc[cpr['sarcastic'] == 1]
cpr = cpr.loc[cpr['condescending'] == 1]
print('Instances where all classes are positive:\t', len(cpr))

Total Training Instances:	 2872
antagonize:	 1467
condescending:	 1575
dismissive:	 1009
hostile:	 923
sarcastic:	 1130
Instances where all classes are negative:	 0
Instances where all classes are positive:	 79


## Prepare UCC testing Dataset

In [18]:
# Examine Test Data
ucc_test.head()

,_unit_id,_trusted_judgments,comment,antagonize,antagonize:confidence,condescending,condescending:confidence,dismissive,dismissive:confidence,hostile,hostile:confidence,sarcastic,sarcastic:confidence
0,1739450989,3,When you have Conservative members now feeling...,0,1.0000,0,1.0000,0,1.0000,0,1.0000,0,1.0
1,1739442069,5,"That's one of the problem, as Germany sent out...",0,0.7955,0,1.0000,0,1.0000,0,1.0000,0,1.0
2,1739464409,3,"Yesshe is, and if she was always and only refe...",0,1.0000,0,1.0000,0,1.0000,0,1.0000,0,1.0
3,1739447549,5,There is nothing honourable about Stephen Harper.,1,0.6215,0,0.5787,0,0.5787,0,0.5948,0,1.0
4,1739466909,3,What a pathetic piece of writing. I have no re...,0,1.0000,0,1.0000,0,1.0000,0,1.0000,0,1.0


In [19]:
# Examine distribution of classes in test set
ucc_antagonize = ucc_test.loc[ucc_test['antagonize'] == 1]
ucc_condescending = ucc_test.loc[ucc_test['condescending'] == 1]
ucc_dismissive = ucc_test.loc[ucc_test['dismissive'] == 1]
ucc_hostile = ucc_test.loc[ucc_test['hostile'] == 1]
ucc_sarcastic = ucc_test.loc[ucc_test['sarcastic'] == 1]

print('antagonize:\t', len(ucc_antagonize))
print('condescending:\t', len(ucc_condescending))
print('dismissive:\t', len(ucc_dismissive))
print('hostile:\t', len(ucc_hostile))
print('sarcastic:\t', len(ucc_sarcastic))
print('\nTotal:\t', len(ucc_test))

all_neg = ucc_test.loc[ucc_test['antagonize'] == 0]
all_neg = all_neg.loc[all_neg['condescending'] == 0]
all_neg = all_neg.loc[all_neg['dismissive'] == 0]
all_neg = all_neg.loc[all_neg['hostile'] == 0]
all_neg = all_neg.loc[all_neg['sarcastic'] == 0]
print('All negative:\t', len(all_neg))
all_pos = ucc_test.loc[ucc_test['antagonize'] == 1]
all_pos = all_pos.loc[all_pos['condescending'] == 1]
all_pos = all_pos.loc[all_pos['dismissive'] == 1]
all_pos = all_pos.loc[all_pos['hostile'] == 1]
all_pos = all_pos.loc[all_pos['sarcastic'] == 1]
print('All positive:\t', len(all_pos))

antagonize:	 203
condescending:	 269
dismissive:	 150
hostile:	 108
sarcastic:	 201

Total:	 4425
All negative:	 3938
All positive:	 13


In [20]:
# Since there are so many instances of 'all negatives', resample to balance the data

# Keep all positive instances. Randomly sample the 'all negative' instances.
np.random.seed(0)
sample_indices = np.random.choice(len(all_neg), size=200, replace=False)
all_neg = all_neg.iloc[sample_indices]

# Combine the sampled 'all negative' instances with the positive instances
ucc_test_balanced = pd.concat([ucc_antagonize, ucc_condescending, ucc_dismissive, ucc_hostile, ucc_sarcastic, all_neg])
ucc_test_balanced.drop_duplicates(inplace=True)
ucc_test_balanced.head()

,_unit_id,_trusted_judgments,comment,antagonize,antagonize:confidence,condescending,condescending:confidence,dismissive,dismissive:confidence,hostile,hostile:confidence,sarcastic,sarcastic:confidence
3,1739447549,5,There is nothing honourable about Stephen Harper.,1,0.6215,0,0.5787,0,0.5787,0,0.5948,0,1.0000
12,1739454509,5,Someone with Boogey's level of education can't...,1,0.8071,1,0.8071,1,0.7916,0,0.5912,0,0.8101
14,2327204319,5,I saw Bibi interviewed on CNN yesterday. He is...,1,0.5863,0,0.7907,0,1.0000,0,0.8106,0,1.0000
19,1739456909,5,"Sheep, it's trump who is trying to kill free s...",1,0.7968,1,0.5880,1,0.7968,0,0.7921,0,0.6035
23,1739465559,5,There is a man in the United States by the nam...,1,0.6052,1,0.6052,1,0.6052,1,0.6052,0,0.7948


In [21]:
# Re-Examine distribution of classes in test set
ucc_antagonize = ucc_test_balanced.loc[ucc_test_balanced['antagonize'] == 1]
ucc_condescending = ucc_test_balanced.loc[ucc_test_balanced['condescending'] == 1]
ucc_dismissive = ucc_test_balanced.loc[ucc_test_balanced['dismissive'] == 1]
ucc_hostile = ucc_test_balanced.loc[ucc_test_balanced['hostile'] == 1]
ucc_sarcastic = ucc_test_balanced.loc[ucc_test_balanced['sarcastic'] == 1]

print('antagonize:\t', len(ucc_antagonize))
print('condescending:\t', len(ucc_condescending))
print('dismissive:\t', len(ucc_dismissive))
print('hostile:\t', len(ucc_hostile))
print('sarcastic:\t', len(ucc_sarcastic))
print('\nTotal:\t', len(ucc_test_balanced))

all_neg = ucc_test_balanced.loc[ucc_test_balanced['antagonize'] == 0]
all_neg = all_neg.loc[all_neg['condescending'] == 0]
all_neg = all_neg.loc[all_neg['dismissive'] == 0]
all_neg = all_neg.loc[all_neg['hostile'] == 0]
all_neg = all_neg.loc[all_neg['sarcastic'] == 0]
print('All negative:\t', len(all_neg))
all_pos = ucc_test_balanced.loc[ucc_test_balanced['antagonize'] == 1]
all_pos = all_pos.loc[all_pos['condescending'] == 1]
all_pos = all_pos.loc[all_pos['dismissive'] == 1]
all_pos = all_pos.loc[all_pos['hostile'] == 1]
all_pos = all_pos.loc[all_pos['sarcastic'] == 1]
print('All positive:\t', len(all_pos))

antagonize:	 203
condescending:	 269
dismissive:	 150
hostile:	 108
sarcastic:	 201

Total:	 687
All negative:	 200
All positive:	 13


In [22]:

ucc_test_balanced_no_scores = ucc_test_balanced.drop(['_unit_id', '_trusted_judgments', 'antagonize:confidence',
                                    'condescending:confidence', 'dismissive:confidence', 'hostile:confidence',
                                    'sarcastic:confidence'], axis='columns')
ucc_test_no_scores = ucc_test.drop(['_unit_id', '_trusted_judgments', 'antagonize:confidence',
                                    'condescending:confidence', 'dismissive:confidence', 'hostile:confidence',
                                    'sarcastic:confidence'], axis='columns')
ucc_test_balanced_no_scores.head()

,comment,antagonize,condescending,dismissive,hostile,sarcastic
3,There is nothing honourable about Stephen Harper.,1,0,0,0,0
12,Someone with Boogey's level of education can't...,1,1,1,0,0
14,I saw Bibi interviewed on CNN yesterday. He is...,1,0,0,0,0
19,"Sheep, it's trump who is trying to kill free s...",1,1,1,0,0
23,There is a man in the United States by the nam...,1,1,1,1,0


In [23]:
# Save test data to files
ucc_test_no_scores.to_csv('/content/drive/My Drive/Colab Notebooks/AISI_project/data/ucc_test_no_scores.csv', index=False)
ucc_test_balanced_no_scores.to_csv('/content/drive/My Drive/Colab Notebooks/AISI_project/data/ucc_test_balanced_no_scores.csv', index=False)

## Prepare UCC validation data

In [ ]:
# Examine distribution of classes in validation set
print('Total validation instances:\t', len(ucc_val))

ucc_antagonize = ucc_val.loc[ucc_val['antagonize'] == 1]
ucc_condescending = ucc_val.loc[ucc_val['condescending'] == 1]
ucc_dismissive = ucc_val.loc[ucc_val['dismissive'] == 1]
ucc_hostile = ucc_val.loc[ucc_val['hostile'] == 1]
ucc_sarcastic = ucc_val.loc[ucc_val['sarcastic'] == 1]

print('antagonize:\t', len(ucc_antagonize))
print('condescending:\t', len(ucc_condescending))
print('dismissive:\t', len(ucc_dismissive))
print('hostile:\t', len(ucc_hostile))
print('sarcastic:\t', len(ucc_sarcastic))

all_neg = ucc_val.loc[ucc_val['antagonize'] == 0]
all_neg = all_neg.loc[all_neg['condescending'] == 0]
all_neg = all_neg.loc[all_neg['dismissive'] == 0]
all_neg = all_neg.loc[all_neg['hostile'] == 0]
all_neg = all_neg.loc[all_neg['sarcastic'] == 0]
print('All negative:\t', len(all_neg))
all_pos = ucc_val.loc[ucc_val['antagonize'] == 1]
all_pos = all_pos.loc[all_pos['condescending'] == 1]
all_pos = all_pos.loc[all_pos['dismissive'] == 1]
all_pos = all_pos.loc[all_pos['hostile'] == 1]
all_pos = all_pos.loc[all_pos['sarcastic'] == 1]
print('All positive:\t', len(all_pos))

Total validation instances:	 4427
antagonize:	 174
condescending:	 238
dismissive:	 143
hostile:	 99
sarcastic:	 195
All negative:	 3964
All positive:	 8


In [ ]:
# Since there are so many instances of 'all negatives', resample to balance the data

# Keep all positive instances. Randomly sample the 'all negative' instances.
np.random.seed(0)
sample_indices = np.random.choice(len(all_neg), size=150, replace=False)
all_neg_df = all_neg.iloc[sample_indices]

# Combine the sampled 'all negative' instances with the positive instances
ucc_val_balanced = pd.concat([ucc_antagonize, ucc_condescending, ucc_dismissive, ucc_hostile, ucc_sarcastic, all_neg_df])
ucc_val_balanced.drop_duplicates(inplace=True)
ucc_val_balanced.head()

,_unit_id,_trusted_judgments,comment,antagonize,antagonize:confidence,condescending,condescending:confidence,dismissive,dismissive:confidence,hostile,hostile:confidence,sarcastic,sarcastic:confidence
86,1739451708,5,Accusations ?Try to Be_A_Real_Human... because...,1,0.8023,1,1.0000,1,0.6052,1,0.8023,1,0.5848
88,2327195888,5,"Freedom, why do you rant here? You don't convi...",1,1.0000,1,0.5973,1,0.8052,0,0.6059,0,1.0000
89,2327195358,5,"What chemical, deoxyribonucleic acid? What are...",1,0.6250,1,0.6250,1,0.5906,0,1.0000,0,0.7932
93,1739447118,5,If you are looking for cuckoos try looking in ...,1,1.0000,1,0.8081,0,0.5855,1,0.8081,1,0.5920
116,2254074868,6,"Wow, the writer is very kind and careful with ...",1,0.8396,1,0.6700,0,0.6711,0,0.5015,1,0.5032


In [ ]:
# Re-examine distribution of classes after balancing
print('Total validation instances:\t', len(ucc_val_balanced))

ucc_antagonize = ucc_val_balanced.loc[ucc_val_balanced['antagonize'] == 1]
ucc_condescending = ucc_val_balanced.loc[ucc_val_balanced['condescending'] == 1]
ucc_dismissive = ucc_val_balanced.loc[ucc_val_balanced['dismissive'] == 1]
ucc_hostile = ucc_val_balanced.loc[ucc_val_balanced['hostile'] == 1]
ucc_sarcastic = ucc_val_balanced.loc[ucc_val_balanced['sarcastic'] == 1]

print('antagonize:\t', len(ucc_antagonize))
print('condescending:\t', len(ucc_condescending))
print('dismissive:\t', len(ucc_dismissive))
print('hostile:\t', len(ucc_hostile))
print('sarcastic:\t', len(ucc_sarcastic))

all_neg = ucc_val_balanced.loc[ucc_val_balanced['antagonize'] == 0]
all_neg = all_neg.loc[all_neg['condescending'] == 0]
all_neg = all_neg.loc[all_neg['dismissive'] == 0]
all_neg = all_neg.loc[all_neg['hostile'] == 0]
all_neg = all_neg.loc[all_neg['sarcastic'] == 0]
print('All negative:\t', len(all_neg))
all_pos = ucc_val_balanced.loc[ucc_val_balanced['antagonize'] == 1]
all_pos = all_pos.loc[all_pos['condescending'] == 1]
all_pos = all_pos.loc[all_pos['dismissive'] == 1]
all_pos = all_pos.loc[all_pos['hostile'] == 1]
all_pos = all_pos.loc[all_pos['sarcastic'] == 1]
print('All positive:\t', len(all_pos))

Total validation instances:	 613
antagonize:	 174
condescending:	 238
dismissive:	 143
hostile:	 99
sarcastic:	 195
All negative:	 150
All positive:	 8


In [ ]:
ucc_val_balanced.head()

,_unit_id,_trusted_judgments,comment,antagonize,antagonize:confidence,condescending,condescending:confidence,dismissive,dismissive:confidence,hostile,hostile:confidence,sarcastic,sarcastic:confidence
86,1739451708,5,Accusations ?Try to Be_A_Real_Human... because...,1,0.8023,1,1.0000,1,0.6052,1,0.8023,1,0.5848
88,2327195888,5,"Freedom, why do you rant here? You don't convi...",1,1.0000,1,0.5973,1,0.8052,0,0.6059,0,1.0000
89,2327195358,5,"What chemical, deoxyribonucleic acid? What are...",1,0.6250,1,0.6250,1,0.5906,0,1.0000,0,0.7932
93,1739447118,5,If you are looking for cuckoos try looking in ...,1,1.0000,1,0.8081,0,0.5855,1,0.8081,1,0.5920
116,2254074868,6,"Wow, the writer is very kind and careful with ...",1,0.8396,1,0.6700,0,0.6711,0,0.5015,1,0.5032


In [ ]:
ucc_val_no_scores = ucc_val.drop(['_unit_id', '_trusted_judgments', 'antagonize:confidence',
                                    'condescending:confidence', 'dismissive:confidence', 'hostile:confidence',
                                    'sarcastic:confidence'], axis='columns')

ucc_val_balanced_no_scores = ucc_val_balanced.drop(['_unit_id', '_trusted_judgments', 'antagonize:confidence',
                                    'condescending:confidence', 'dismissive:confidence', 'hostile:confidence',
                                    'sarcastic:confidence'], axis='columns')
ucc_val_no_scores.head()

,comment,antagonize,condescending,dismissive,hostile,sarcastic
0,"That's exactly what they've done. Beck, Toront...",0,0,0,0,0
1,Should we really pick a voting system that ens...,0,0,0,0,0
2,"actually, the president of the united states h...",0,0,0,0,0
3,"There is no aura of power, there only is irres...",0,0,0,0,0
4,Delusional much? I think you had better do som...,0,1,0,0,0


In [ ]:
# Save validation data to files
ucc_val_no_scores.to_csv('/content/drive/My Drive/Colab Notebooks/AISI_project/data/ucc_val_no_scores.csv', index=False)
ucc_val_balanced_no_scores.to_csv('/content/drive/My Drive/Colab Notebooks/AISI_project/data/ucc_val_balanced_no_scores.csv', index=False)

# Reddit Abuse Dataset



## Examine Reddit Abuse Dataset shelves

In [ ]:
reddit_data_dir = '/content/drive/My Drive/Colab Notebooks/AISI_project/og_datasets/reddit_data_shelves/'

# Load Reddit Abuse Posts dataset
shelf1 = shelve.open(reddit_data_dir+'redditAbuseSubmissions')
shelf1_keys = list(shelf1.keys())
print('Shelf 1 keys: ', shelf1_keys)
shelf1_dict = {}
for key in shelf1_keys:
    shelf1_dict[key] = shelf1[key]
shelf1.close()
shelf1_df = pd.DataFrame(shelf1_dict)

shelf2 = shelve.open(reddit_data_dir+'redditAbuseComments')
shelf2_keys = list(shelf2.keys())
print('Shelf 2 keys: ', shelf2_keys)
shelf2_dict = {}
for key in shelf2_keys:
    shelf2_dict[key] = shelf2[key]
shelf2.close()
shelf2_df = pd.DataFrame(shelf2_dict)

shelf3 = shelve.open(reddit_data_dir+'redditRelationshipsData')
shelf3_keys = list(shelf3.keys())
print('Shelf 3 keys: ', shelf3_keys)
shelf3_dict = {}
for key in shelf3_keys:
    shelf3_dict[key] = shelf3[key]
shelf3.close()
shelf3_df = pd.DataFrame(shelf3_dict)

shelf4 = shelve.open(reddit_data_dir+'redditAbuseUneven')
shelf4_keys = list(shelf4.keys())
print('Shelf 4 keys: ', shelf4_keys)
shelf4_train_dict = {}
shelf4_train_dict['text'] = shelf4['XTrain']
shelf4_train_dict['labels'] = shelf4['labelsTrain']
shelf4_test_dict = {}
shelf4_test_dict['text'] = shelf4['XTest']
shelf4_test_dict['labels'] = shelf4['labelsTest']
shelf4_train_df = pd.DataFrame(shelf4_train_dict)
shelf4_test_df = pd.DataFrame(shelf4_test_dict)


print('\nReddit Abuse Submissions')
print(shelf1_df.head(), '\n\n')
print('Reddit Abuse Comments')
print(shelf2_df.head(), '\n\n')
print('Reddit Relationship Advice')
print(shelf3_df.head(), '\n\n')
print(shelf3_df.iloc[4]['data'], '\n\n')
print('Reddit Abuse Uneven Set')
print(shelf4_train_df.head(), '\n\n')
print(shelf4_test_df.head(), '\n\n')


Shelf 1 keys:  ['data', 'subIds', 'roles', 'predicates', 'labels']
Shelf 2 keys:  ['commLabels', 'commData']
Shelf 3 keys:  ['subIds', 'data']
Shelf 4 keys:  ['subIdsTest', 'labelsTest', 'XTrain', 'XTest', 'labelsTrain', 'subIdsTrain']

Reddit Abuse Submissions
                                                data  subIds  \
0  I cant eat pls help I need help\nMy anxiety ha...  2wjl43   
1  Financial Independence I am 18, with no job an...  2tdh8q   
2  Who decided that online calculus assignments w...  2vwei8   
3  My friend recently told me she was abused as a...   p013r   
4  How's it going on this monday night? I am list...  2xrdhg   

                                               roles  \
0  [am-adv, am-adv, am-adv, am-dir, am-dis, am-mn...   
1  [am-mnr, am-mod, am-rec, am-tmp, am-tmp, taker...   
2  [am-loc, causer, frustrater, comment, decision...   
3  [am-cau, am-cau, am-cau, am-dir, am-dir, am-di...   
4  [am-mnr, am-rec, am-tmp, entity in motion/goer...   

                

## Balance Abuse and Non-Abuse instances

In [ ]:
# Examine the distribution of Abuse and Non-Abuse Instances in the training data
reddit_train_df = shelf4_train_df
reddit_test_df = shelf4_test_df

print(reddit_train_df.head())
print('\nTotal instances for Training: ', len(reddit_train_df))
print('Abuse instances for Training: ', len(reddit_train_df[reddit_train_df['labels'] == 'abuse']))
print('Non-Abuse instances for Training: ', len(reddit_train_df[reddit_train_df['labels'] == 'non_abuse']))

print('\nTotal instances for Testing: ', len(reddit_test_df))
print('Abuse instances for Testing: ', len(reddit_test_df[reddit_test_df['labels'] == 'abuse']))
print('Non-Abuse instances for Testing: ', len(reddit_test_df[reddit_test_df['labels'] == 'non_abuse']))

                                                text     labels
0  car backed into mine. insurance seeking liabil...  non_abuse
1  in 3 months i start working my dream job. i re...  non_abuse
2  it sucks to be a man sometimes. my ex-girlfrie...      abuse
3  what issue is very close to your heart? and ho...  non_abuse
4  how do you get out of that routine of "all-or-...  non_abuse

Total instances for Training:  15602
Abuse instances for Training:  1131
Non-Abuse instances for Training:  14471

Total instances for Testing:  2754
Abuse instances for Testing:  205
Non-Abuse instances for Testing:  2549


In [ ]:
# Since the number of Abuse and non-abuse instances in the training data is highly imbalanced,
# randomly resample the non-abuse instances to help balance the dataset.
abuse_instances = reddit_train_df[reddit_train_df['labels'] == 'abuse']
nonabuse_instances = reddit_train_df[reddit_train_df['labels'] == 'non_abuse']

# Sample negative non-abuse instances
sample_indices = np.random.choice(len(nonabuse_instances), size=1150, replace=False)
sampled_instances = nonabuse_instances.iloc[sample_indices]

# Recreate the dataset
reddit_train_balanced = pd.concat([abuse_instances, sampled_instances])
reddit_train_balanced.drop_duplicates(inplace=True)

# Re-examine balance of classes
pos_df = reddit_train_balanced[reddit_train_balanced['labels'] == 'abuse']
neg_df = reddit_train_balanced[reddit_train_balanced['labels'] == 'non_abuse']
print("Abuse: %d\t(%.2f%%)\t\t Non-Abuse: %d\t(%.2f%%)"
        % (len(pos_df), 100*len(pos_df)/len(reddit_train_balanced), len(neg_df), 100*len(neg_df)/len(reddit_train_balanced)))


Abuse: 1131	(49.58%)		 Non-Abuse: 1150	(50.42%)


In [ ]:
# Repeat for Testing data
abuse_instances = reddit_test_df[reddit_test_df['labels'] == 'abuse']
nonabuse_instances = reddit_test_df[reddit_test_df['labels'] == 'non_abuse']

# Sample negative non-abuse instances
sample_indices = np.random.choice(len(nonabuse_instances), size=205, replace=False)
sampled_instances = nonabuse_instances.iloc[sample_indices]

# Recreate the dataset
reddit_test_balanced = pd.concat([abuse_instances, sampled_instances])
reddit_test_balanced.drop_duplicates(inplace=True)

# Re-examine balance of classes
pos_df = reddit_test_balanced[reddit_test_balanced['labels'] == 'abuse']
neg_df = reddit_test_balanced[reddit_test_balanced['labels'] == 'non_abuse']
print("Abuse: %d\t(%.2f%%)\t\t Non-Abuse: %d\t(%.2f%%)"
        % (len(pos_df), 100*len(pos_df)/len(reddit_test_balanced), len(neg_df), 100*len(neg_df)/len(reddit_test_balanced)))

Abuse: 205	(50.00%)		 Non-Abuse: 205	(50.00%)


In [ ]:
reddit_train_df.to_csv('/content/drive/My Drive/Colab Notebooks/AISI_project/data/reddit_train.csv', index=False)
reddit_train_balanced.to_csv('/content/drive/My Drive/Colab Notebooks/AISI_project/data/reddit_train_balanced.csv', index=False)

reddit_test_df.to_csv('/content/drive/My Drive/Colab Notebooks/AISI_project/data/reddit_test.csv', index=False)
reddit_test_balanced.to_csv('/content/drive/My Drive/Colab Notebooks/AISI_project/data/reddit_test_balanced.csv', index=False)